# Demo 2: Building Predictive Models with Health Metrics

In this demo, we'll build and compare different predictive models for health time series data. We'll cover:
1. Feature engineering for time series
2. Time series cross-validation
3. Model training and evaluation
4. Making predictions

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
plt.style.use('seaborn')
sns.set_palette('husl')

## 1. Generate Sample Health Data

Let's create a synthetic dataset of patient vital signs with more complex patterns.

In [ ]:
# Create date range for one month of hourly measurements
dates = pd.date_range(start='2024-01-01', end='2024-01-31', freq='H')

# Create more complex patterns
hour_effect = 10 * np.sin(2 * np.pi * (dates.hour - 6) / 24)
day_effect = 5 * np.sin(2 * np.pi * dates.dayofweek / 7)
trend = np.linspace(0, 5, len(dates))

# Generate vital signs with multiple patterns
vitals = pd.DataFrame({
    'timestamp': dates,
    'heart_rate': 70 + hour_effect + day_effect + trend + np.random.normal(0, 3, len(dates)),
    'temperature': 37 + 0.3 * np.sin(2 * np.pi * (dates.hour - 4) / 24) + 0.1 * np.sin(2 * np.pi * dates.dayofweek / 7) + np.random.normal(0, 0.1, len(dates)),
    'blood_pressure': 120 + 5 * np.sin(2 * np.pi * (dates.hour - 8) / 24) + 2 * np.sin(2 * np.pi * dates.dayofweek / 7) + np.random.normal(0, 2, len(dates))
})

vitals = vitals.set_index('timestamp')
vitals.head()

## 2. Feature Engineering

Create relevant features for time series prediction.

In [ ]:
def create_features(df, target_col, window_sizes=[3, 6, 12, 24]):
    """Create time series features from datetime index."""
    df = df.copy()
    
    # Time-based features
    df['hour'] = df.index.hour
    df['dayofweek'] = df.index.dayofweek
    df['quarter'] = df.index.quarter
    df['month'] = df.index.month
    df['year'] = df.index.year
    df['dayofyear'] = df.index.dayofyear
    
    # Cyclical features
    df['hour_sin'] = np.sin(2 * np.pi * df.index.hour/24)
    df['hour_cos'] = np.cos(2 * np.pi * df.index.hour/24)
    
    # Lag features
    df['lag_1'] = df[target_col].shift(1)
    df['lag_2'] = df[target_col].shift(2)
    df['lag_3'] = df[target_col].shift(3)
    
    # Rolling window features
    for window in window_sizes:
        df[f'rolling_mean_{window}'] = df[target_col].rolling(window=window).mean()
        df[f'rolling_std_{window}'] = df[target_col].rolling(window=window).std()
    
    return df

# Create features for heart rate prediction
features_df = create_features(vitals, 'heart_rate')

# Drop rows with NaN values (from lag/rolling features)
features_df = features_df.dropna()

# Display feature correlations with target
correlations = features_df.corr()['heart_rate'].sort_values(ascending=False)
plt.figure(figsize=(10, 6))
correlations.plot(kind='bar')
plt.title('Feature Correlations with Heart Rate')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Time Series Cross-Validation

Set up time series cross-validation to properly evaluate our models.

In [ ]:
# Prepare features and target
feature_columns = [col for col in features_df.columns if col != 'heart_rate']
X = features_df[feature_columns]
y = features_df['heart_rate']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

# Initialize TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)

# Visualize the splits
fig, ax = plt.subplots(figsize=(10, 5))
for i, (train_idx, test_idx) in enumerate(tscv.split(X_scaled)):
    ax.scatter(train_idx, [i] * len(train_idx), c='blue', marker='_', label='Train' if i==0 else '')
    ax.scatter(test_idx, [i] * len(test_idx), c='red', marker='_', label='Test' if i==0 else '')

ax.set_title('Time Series Cross-Validation Splits')
ax.legend()
plt.show()

## 4. Model Training and Evaluation

Train and compare different models using our time series cross-validation setup.

In [ ]:
def evaluate_model(model, X, y, cv):
    """Evaluate model using time series cross-validation."""
    mae_scores = []
    rmse_scores = []
    r2_scores = []
    
    for train_idx, test_idx in cv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        mae_scores.append(mean_absolute_error(y_test, y_pred))
        rmse_scores.append(np.sqrt(mean_squared_error(y_test, y_pred)))
        r2_scores.append(r2_score(y_test, y_pred))
    
    return {
        'mae': np.mean(mae_scores),
        'rmse': np.mean(rmse_scores),
        'r2': np.mean(r2_scores)
    }

# Initialize models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

# Evaluate models
results = {}
for name, model in models.items():
    results[name] = evaluate_model(model, X_scaled, y, tscv)

# Display results
results_df = pd.DataFrame(results).T
print("Model Evaluation Results:")
print(results_df)

# Plot results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = ['mae', 'rmse', 'r2']
titles = ['Mean Absolute Error', 'Root Mean Squared Error', 'R-squared']

for ax, metric, title in zip(axes, metrics, titles):
    results_df[metric].plot(kind='bar', ax=ax)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Making Predictions

Use the best model to make predictions and visualize the results.

In [ ]:
# Train final model on all data except last week
train_end = vitals.index[-24*7]  # Last week for testing
train_features = X_scaled[X_scaled.index <= train_end]
test_features = X_scaled[X_scaled.index > train_end]
train_target = y[y.index <= train_end]
test_target = y[y.index > train_end]

# Use Random Forest as our final model
final_model = RandomForestRegressor(n_estimators=100, random_state=42)
final_model.fit(train_features, train_target)

# Make predictions
train_pred = final_model.predict(train_features)
test_pred = final_model.predict(test_features)

# Plot results
plt.figure(figsize=(15, 6))
plt.plot(train_target.index, train_target, label='Actual (Train)', alpha=0.5)
plt.plot(train_target.index, train_pred, label='Predicted (Train)', alpha=0.5)
plt.plot(test_target.index, test_target, label='Actual (Test)', alpha=0.5)
plt.plot(test_target.index, test_pred, label='Predicted (Test)', alpha=0.5)
plt.title('Heart Rate Predictions')
plt.legend()
plt.tight_layout()
plt.show()

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance.head(10), x='importance', y='feature')
plt.title('Top 10 Most Important Features')
plt.tight_layout()
plt.show()